In [ ]:
# KG1 Nemotron - H100 one-cell pre-score calibration
# Nao faz submit. Objetivo: rodar o pre-score local calibrado contra anchors Kaggle conhecidos.

import os, json, shutil, zipfile, subprocess, sys, textwrap, importlib.util, inspect
from pathlib import Path

# =========================
# Configuracao principal
# =========================
DRIVE_DIR = Path("/content/drive/MyDrive/KG1_Nemotron_prescore")
WORK_DIR = Path("/content/kg1_prescore")
PACK_ZIP = DRIVE_DIR / "kg1_prescore_calibration_pack.zip"
ADAPTERS_ROOT = DRIVE_DIR / "adapters"
OUT_DIR = DRIVE_DIR / "prescore_outputs_h100"

# H100 preset: aumente N_SAMPLES para 180-300 quando os anchors estiverem calibrando bem.
N_SAMPLES = 120
MAX_ADAPTERS = 8
MAX_NEW_TOKENS = 1024
SCORE_MODE = "raw"
GPU_MEMORY_MARGIN_GB = 8
USE_CACHE = True
DATASET_RELATIVE = "data/v94/v94_equation_crypt_val.jsonl"
FINGERPRINT_MODE = "partial"  # partial evita minutos lendo SHA completo de zips/adapters no Drive.
DELETE_ANCHOR_ZIPS_AFTER_EXTRACT = True

DEFAULT_ANCHORS = {
    "canonical_086": {
        "hf_file": "data/v87_delta/canonical_086_submission.zip",
        "kaggle_score": 0.86,
    },
    "v088_084": {
        "hf_file": "prescore/anchors/v088_084_submission.zip",
        "kaggle_score": 0.84,
    },
}

def sh(cmd: str, check: bool = True):
    print("\n$ " + cmd)
    p = subprocess.run(cmd, shell=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")
    return p.returncode

print("=== 1. GPU / Python ===")
sh("nvidia-smi", check=False)
print(sys.version)

print("\n=== 2. Install dependencies ===")
sh('pip -q install -U "transformers==5.3.0" "peft==0.18.1" accelerate bitsandbytes safetensors huggingface_hub sentencepiece tqdm kaggle "jedi>=0.16"')
sh('pip -q install -U "torchao>=0.16.0"')
sh("pip -q install -U packaging ninja wheel setuptools")
sh('MAX_JOBS=4 pip -q install -U "causal-conv1d>=1.4.0" "mamba-ssm>=2.2.4" --no-build-isolation')

import transformers, torchao
print("torchao:", getattr(torchao, "__version__", "unknown"))
print("transformers:", transformers.__version__)
assert hasattr(transformers, "WeightConverter"), "transformers sem WeightConverter; precisa de transformers 5.x"
assert "distributed_operation" in str(inspect.signature(transformers.WeightConverter)), inspect.signature(transformers.WeightConverter)
assert importlib.util.find_spec("mamba_ssm") is not None, "mamba_ssm nao foi instalado"

print("\n=== 3. Mount Drive / secrets ===")
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    def secret(name, default=""):
        try:
            return userdata.get(name) or default
        except Exception:
            return default
except Exception:
    userdata = None
    def secret(name, default=""):
        return os.environ.get(name, default)

HF_TOKEN = secret("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
KAGGLE_USERNAME = secret("KAGGLE_USERNAME", os.environ.get("KAGGLE_USERNAME", ""))
KAGGLE_KEY = secret("KAGGLE_KEY", os.environ.get("KAGGLE_KEY", ""))

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY

WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
ADAPTERS_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR)
print("DRIVE_DIR:", DRIVE_DIR)
print("ADAPTERS_ROOT:", ADAPTERS_ROOT)
print("OUT_DIR:", OUT_DIR)

print("\n=== 4. Download/extract latest pre-score pack ===")
from huggingface_hub import hf_hub_download
downloaded_pack = hf_hub_download(
    repo_id="felipesp1983/kg1-nemotron-training",
    repo_type="dataset",
    filename="prescore/kg1_prescore_calibration_pack.zip",
    token=HF_TOKEN or None,
    force_download=True,
)
shutil.copy2(downloaded_pack, PACK_ZIP)
print("Pack salvo em:", PACK_ZIP, "size=", PACK_ZIP.stat().st_size)

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK_ZIP) as zf:
    zf.extractall(WORK_DIR)
print("Pack extraido.")
sh("find /content/kg1_prescore -maxdepth 3 -type f | head -80", check=False)

print("\n=== 5. Discover/download adapters ===")
def extract_adapter_zips():
    extracted = []
    for zip_path in ADAPTERS_ROOT.rglob("*.zip"):
        out = zip_path.with_suffix("")
        out.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            if "adapter_config.json" in names and "adapter_model.safetensors" in names:
                zf.extract("adapter_config.json", out)
                zf.extract("adapter_model.safetensors", out)
                extracted.append(out)
                print("extraido:", zip_path, "->", out)
    return extracted

def cleanup_extracted_anchor_zips():
    if not DELETE_ANCHOR_ZIPS_AFTER_EXTRACT:
        return
    freed = 0
    for name in DEFAULT_ANCHORS:
        z = ADAPTERS_ROOT / f"{name}.zip"
        extracted_model = ADAPTERS_ROOT / name / "adapter_model.safetensors"
        if z.exists() and extracted_model.exists():
            size = z.stat().st_size
            z.unlink()
            freed += size
            print("zip removido apos extracao:", z, "freed_gb=", round(size / 1024**3, 3))
    if freed:
        print("Total liberado removendo zips de anchors:", round(freed / 1024**3, 3), "GB")

def find_adapter_dirs():
    out = []
    for cfg in ADAPTERS_ROOT.rglob("adapter_config.json"):
        if (cfg.parent / "adapter_model.safetensors").exists():
            out.append(cfg.parent)
    return sorted(set(out))

extract_adapter_zips()
adapter_dirs = find_adapter_dirs()

if not adapter_dirs:
    print("Nenhum adapter encontrado; baixando anchors padrao do HF...")
    for name, spec in DEFAULT_ANCHORS.items():
        dst = ADAPTERS_ROOT / f"{name}.zip"
        if not dst.exists():
            downloaded = hf_hub_download(
                repo_id="felipesp1983/kg1-nemotron-training",
                repo_type="dataset",
                filename=spec["hf_file"],
                token=HF_TOKEN or None,
            )
            shutil.copy2(downloaded, dst)
            print("anchor baixado:", name, "->", dst)
    extract_adapter_zips()
    adapter_dirs = find_adapter_dirs()

cleanup_extracted_anchor_zips()

print("Adapters encontrados:", len(adapter_dirs))
for p in adapter_dirs[:50]:
    print("-", p)
assert adapter_dirs, "Nenhum adapter encontrado/baixado"

print("\n=== 6. Calibration pairs / holdout ===")
CALIBRATION_PAIRS = {}
for name, spec in DEFAULT_ANCHORS.items():
    candidate_dir = ADAPTERS_ROOT / name
    if (candidate_dir / "adapter_config.json").exists():
        CALIBRATION_PAIRS[str(candidate_dir)] = spec["kaggle_score"]

calibration_path = OUT_DIR / "calibration_pairs.json"
calibration_path.write_text(json.dumps(CALIBRATION_PAIRS, indent=2, sort_keys=True), encoding="utf-8")
print(calibration_path)
print(json.dumps(CALIBRATION_PAIRS, indent=2, sort_keys=True))
assert len(CALIBRATION_PAIRS) >= 2, "Calibracao minima precisa dos anchors canonical_086 e v088_084"

DATASET_PATH = WORK_DIR / DATASET_RELATIVE
assert DATASET_PATH.exists(), DATASET_PATH
print(DATASET_PATH, "lines=", sum(1 for _ in DATASET_PATH.open(encoding="utf-8")))

print("\n=== 7. H100 pre-score run ===")
cmd = f"""
PYTHONUNBUFFERED=1 python -u /content/kg1_prescore/scripts/batch_validate_prescore.py \
  --adapters-root "{ADAPTERS_ROOT}" \
  --dataset-path "{DATASET_PATH}" \
  --calibration-pairs "{calibration_path}" \
  --n-samples {N_SAMPLES} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --score-mode {SCORE_MODE} \
  --output-dir "{OUT_DIR}" \
  --max-adapters {MAX_ADAPTERS} \
  --fingerprint-mode {FINGERPRINT_MODE} \
  --gpu-memory-margin-gb {GPU_MEMORY_MARGIN_GB} \
  {"--use-cache" if USE_CACHE else ""}
"""
print(cmd)
sh(cmd)

print("\n=== 8. Outputs ===")
for path in sorted(OUT_DIR.glob("*.json")):
    print("\n###", path)
    txt = path.read_text(encoding="utf-8", errors="replace")
    print(txt[:6000])

print("\n=== DONE ===")
print("Nao use score local bruto como Kaggle score. Use apenas como gate calibrado contra anchors conhecidos.")
